## spleeter 安裝
使用 spleeter 進行音源分離


In [1]:
!pip install spleeter

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [2]:
# 產生 output 資料夾儲存分出來的 人聲（vocals） 和 配樂（accompaniment）
!spleeter separate -p spleeter:2stems -o output vocals_mix_1.wav

INFO:spleeter:File output/vocals_mix_1/vocals.wav written succesfully
INFO:spleeter:File output/vocals_mix_1/accompaniment.wav written succesfully


## pyAudioAnalysis 安裝
使用 pyAudioAnalysis 進行性別分類

In [3]:
!git clone https://github.com/tyiannak/pyAudioAnalysis.git
!pip install -r ./pyAudioAnalysis/requirements.txt 

fatal: destination path 'pyAudioAnalysis' already exists and is not an empty directory.
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [4]:
!cd pyAudioAnalysis && pip install -e .

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Obtaining file:///workspace/speeter/pyAudioAnalysis
  Preparing metadata (setup.py) ... done
  Attempting uninstall: pyAudioAnalysis
    Found existing installation: pyAudioAnalysis 0.3.14
    Uninstalling pyAudioAnalysis-0.3.14:
      Successfully uninstalled pyAudioAnalysis-0.3.14
  DEPRECATION: Legacy editable install of pyAudioAnalysis==0.3.14 from file:///workspace/speeter/pyAudioAnalysis (setup.py develop) is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to add a pyproject.toml or enable --use-pep517, and use setuptools >= 64. If the resulting installation is not behaving as expected, try using --config-settings editable_mode=compat. Please consult the setuptools documentation for more information. Discussion can be found at https://github.com/pypa/pip/issues/11457
  Running setup.py develop for pyAudioAnalysis


### 測試一：男女音重疊

In [5]:
from IPython.display import Audio
Audio('vocals_mix_1.wav')

以上 wav檔案直接給入 pyAudioAnalysis ，會出現以下訊息
*ValueError: File format b'ID3\x04' not understood. Only 'RIFF', 'RIFX', and 'RF64' supported.*

--> 透過 ffmpeg 轉換
該錯誤訊息表示此 WAV 文件實際上可能包含 ID3 標籤（通常用於 MP3 文件），導致該套件無法識別其格式。

可以：
1. 使用 ffmpeg（最快最有效）
2. 使用 pydub 轉換（適合 Python）
3. 使用 wave 重新寫入（如果 WAV 只是標頭異常）
4. 手動刪除 ID3 標籤（如果確定是 ID3 問題）
如果不確定，你可以用 ffmpeg -i input.wav 先查看文件的詳細信息！

In [6]:
!ffmpeg -i vocals_mix_1.wav -acodec pcm_s16le -ar 44100 vocals_mix_1_test.wav

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [7]:
# ### 補充 方法 2：使用　python 轉換
# # !pip install pydub
# from pydub import AudioSegment

# # 讀取原始 WAV 檔案
# audio = AudioSegment.from_file("input.wav")

# # 轉換並儲存為標準 WAV 格式
# audio.export("output.wav", format="wav")

In [8]:
# #### 使用 wave 重新寫入
# ##   如果你確定音頻數據是 PCM 編碼，但格式標頭有問題，你可以使用 wave 模組來重新寫入：

# import wave

# with wave.open("input.wav", "rb") as infile:
#     params = infile.getparams()  # 取得音訊參數
#     audio_data = infile.readframes(infile.getnframes())

# with wave.open("output.wav", "wb") as outfile:
#     outfile.setparams(params)  # 設定相同參數
#     outfile.writeframes(audio_data)  # 寫入數據

# ## 這樣會重新儲存 WAV 檔案，可能會修正格式錯誤。


In [9]:
# ### 補充 方法 3：手動檢查並修正
# 如果你懷疑 WAV 文件的前幾個字節包含 ID3 標籤，可以手動刪除 ID3 標頭：

# !tail -c +128 input.wav > output.wav

## 這將刪除前 128 個字節（常見的 ID3v1 標籤大小），然後儲存為 output.wav。

In [10]:
from pydub import AudioSegment
from pyAudioAnalysis import audioSegmentation as aS

labels = aS.speaker_diarization("vocals_mix_1_test.wav", n_speakers=2)
print(labels)

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 0.24.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


(array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), -1, -1)


`aS.speaker_diarization("vocals.wav", n_speakers=2)` 不會直接輸出音檔，而是返回 一個數組（array），

其中包含 每個時間片段的說話人標籤。
* 0 代表第一位說話者（可能是男聲）
* 1 代表第二位說話者（可能是女聲）
* -1 代表無法確定說話者的部分（靜音或背景聲）

#### 將不同說話者的音訊分開
目前 `aS.speaker_diarization()` 只輸出說話人標籤，不會自動分割音訊。因此，需要手動根據標籤來分離男聲和女聲。

In [12]:
from pydub import AudioSegment
import numpy as np

label_list = labels[0] # 確保是一維數組


# 讀取音檔
audio = AudioSegment.from_wav("vocals_mix_1_test.wav")
frame_rate = len(audio) / len(label_list)  # 每個標籤對應的音訊時間

# 初始化兩個音訊段
voice1_audio = AudioSegment.silent(duration=0)
voice2_audio = AudioSegment.silent(duration=0)

# 根據標籤提取音訊
for i, label in enumerate(label_list):
    start_time = int(i * frame_rate)  # 計算該片段的開始時間
    end_time = int((i + 1) * frame_rate)  # 計算該片段的結束時間
    segment = audio[start_time:end_time]

    if label == 0:  # 第一位說話者
        voice1_audio += segment
    elif label == 1:  # 第二位說話者
        voice2_audio += segment

# 儲存結果
voice1_audio.export("voice1_demo1.wav", format="wav")
voice2_audio.export("voice2_demo1.wav", format="wav")

print("分離完成，輸出 voice1_demo1.wav 和 voice2_demo1.wav")


分離完成，輸出 voice1_demo1.wav 和 voice2_demo1.wav


In [13]:
from IPython.display import Audio

Audio('voice1_demo1.wav')

In [14]:
from IPython.display import Audio

Audio('voice2_demo1.wav')

兩聲道重疊聽起來是有點分不開

### 測試二：男女音分開

In [15]:
!ffmpeg -i vocals_mix_2.wav -acodec pcm_s16le -ar 44100 vocals_mix_2_test.wav

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [24]:
from pydub import AudioSegment
from pyAudioAnalysis import audioSegmentation as aS

labels = aS.speaker_diarization("vocals_mix_2_test.wav", n_speakers=2)
print(labels)

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 0.24.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


(array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), -1, -1)


In [25]:
from pydub import AudioSegment
import numpy as np

label_list = labels[0] # 確保是一維數組


# 讀取音檔
audio = AudioSegment.from_wav("vocals_mix_2_test.wav")
frame_rate = len(audio) / len(label_list)  # 每個標籤對應的音訊時間

# 初始化兩個音訊段
voice1_audio = AudioSegment.silent(duration=0)
voice2_audio = AudioSegment.silent(duration=0)

# 根據標籤提取音訊
for i, label in enumerate(label_list):
    start_time = int(i * frame_rate)  # 計算該片段的開始時間
    end_time = int((i + 1) * frame_rate)  # 計算該片段的結束時間
    segment = audio[start_time:end_time]

    if label == 0:  # 第一位說話者
        voice1_audio += segment
    elif label == 1:  # 第二位說話者
        voice2_audio += segment

# 儲存結果
voice1_audio.export("voice1_demo2.wav", format="wav")
voice2_audio.export("voice2_demo2.wav", format="wav")

print("分離完成，輸出 voice1_demo2.wav 和 voice2_demo2.wav")


分離完成，輸出 voice1_demo2.wav 和 voice2_demo2.wav


### 測試結果

In [26]:
from IPython.display import Audio

Audio('voice1_demo2.wav')

In [27]:
from IPython.display import Audio

Audio('voice2_demo2.wav')

**Note**: 測試過多次，每次產生的結果不太一樣